In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

# 1. Load & Prepare Dataset (0 = Malignant, 1 = Benign)
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# 2. Feature Selection (Top 10 Clinical Biomarkers)
selector = SelectKBest(score_func=f_classif, k=10)
X_selected = selector.fit_transform(X, y)
selected_features = X.columns[selector.get_support()].tolist()

# 3. Train-Test Split & Standardization
X_train, X_test, y_train, y_test = train_test_split(X_selected, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Train & Automatically Select Best Model
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

best_model = None
best_model_name = ""
best_f1 = 0

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    f1 = f1_score(y_test, preds)
    if f1 > best_f1:
        best_f1 = f1
        best_model = model
        best_model_name = name

# 5. Dynamic Single-Patient Diagnostic Function
def diagnose_single_patient(patient_id, custom_measurements=None):
    """
    Dynamically creates a single patient profile from baseline median values,
    updates it with any custom measurements provided, and outputs one clinical report.
    """
    # Start with baseline median values across all 30 features
    patient_profile = X.median().to_dict()

    # Dynamically update with the single patient's specific measurements
    if custom_measurements:
        for feature, value in custom_measurements.items():
            if feature in patient_profile:
                patient_profile[feature] = value
            else:
                print(f"Warning: '{feature}' is not a valid feature name and was skipped.")

    # Transform & Scale single patient input
    patient_df = pd.DataFrame([patient_profile], columns=data.feature_names)
    patient_sel = selector.transform(patient_df)
    patient_scaled = scaler.transform(patient_sel)

    # Predict probabilities (0 = Malignant, 1 = Benign)
    probs = best_model.predict_proba(patient_scaled)[0]
    malignant_prob = probs[0] * 100
    benign_prob = probs[1] * 100

    # Risk Stratification & Clinical Action
    if malignant_prob >= 75:
        diagnosis = "MALIGNANT (High Suspicion)"
        risk_tier = "HIGH RISK (Tier 3)"
        action = "Urgent oncology referral and confirmatory core needle biopsy recommended."
    elif malignant_prob >= 30:
        diagnosis = "INDETERMINATE / BORDERLINE"
        risk_tier = "MODERATE RISK (Tier 2)"
        action = "Follow-up diagnostic imaging (ultrasound/MRI) and specialist review within 2 weeks."
    else:
        diagnosis = "BENIGN (Non-Cancerous)"
        risk_tier = "LOW RISK (Tier 1)"
        action = "Routine annual screening; no immediate invasive intervention indicated."

    # Identify top 3 abnormal biomarkers for this patient
    z_scores = np.abs(patient_scaled[0])
    top_idx = np.argsort(z_scores)[::-1][:3]
    key_drivers = [f"{selected_features[i]} ({patient_sel[0][i]:.2f})" for i in top_idx]

    print("=" * 64)
    print(f"SINGLE-PATIENT DIAGNOSTIC REPORT | Patient ID: {patient_id}")
    print(f"Active Model: {best_model_name} (Validation F1-Score: {best_f1:.4f})")
    print("=" * 64)
    print(f"Primary Assessment    : {diagnosis}")
    print(f"Malignancy Probability: {malignant_prob:.1f}%  |  Benign Probability: {benign_prob:.1f}%")
    print(f"Risk Stratification   : {risk_tier}")
    print(f"Key Biomarker Drivers : {', '.join(key_drivers)}")
    print(f"Recommended Action    : {action}")
    print("=" * 64)


# --- DYNAMIC SINGLE-PATIENT INPUT ---
# Pass any dictionary of measurements for ONE patient (e.g., high-risk biopsy values)
single_patient_input = {
    'mean radius': 19.5,
    'mean perimeter': 128.0,
    'mean area': 1180.0,
    'mean concavity': 0.22,
    'worst radius': 24.1,
    'worst perimeter': 158.0,
    'worst area': 1750.0,
    'worst concave points': 0.24
}

diagnose_single_patient(patient_id="PT-2048", custom_measurements=single_patient_input)

SINGLE-PATIENT DIAGNOSTIC REPORT | Patient ID: PT-2048
Active Model: Logistic Regression (Validation F1-Score: 0.9787)
Primary Assessment    : MALIGNANT (High Suspicion)
Malignancy Probability: 99.9%  |  Benign Probability: 0.1%
Risk Stratification   : HIGH RISK (Tier 3)
Key Biomarker Drivers : worst concave points (0.24), mean concavity (0.22), worst radius (24.10)
Recommended Action    : Urgent oncology referral and confirmatory core needle biopsy recommended.
